# HOG Feature Exploration

This notebook visualises HOG descriptors extracted from character patches to help tune `configs/features.yaml`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from skimage import exposure

from src.features.hog_extractor import extract_hog

## 1. Synthetic character patch

In [ ]:
# Create a simple synthetic 'A'-like blob
rng = np.random.default_rng(42)
patch = np.zeros((32, 32), dtype=np.uint8)
patch[4:28, 12:20] = 220   # vertical stroke
patch[4:10, 6:26] = 220    # top bar
patch[14:18, 6:26] = 180   # cross bar

plt.figure(figsize=(3, 3))
plt.imshow(patch, cmap='gray')
plt.title('Synthetic character patch')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Extract HOG descriptor and visualise gradient magnitudes

In [ ]:
fd, hog_img = hog(
    patch,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm='L2-Hys',
    visualize=True,
)

hog_img_rescaled = exposure.rescale_intensity(hog_img, in_range=(0, 10))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(patch, cmap='gray')
axes[0].set_title('Original patch')
axes[1].imshow(hog_img_rescaled, cmap='gray')
axes[1].set_title('HOG visualisation')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'Descriptor length: {fd.shape[0]}')

## 3. Compare descriptor lengths for different configurations

In [ ]:
configs = [
    dict(orientations=9,  pixels_per_cell=(8, 8),  cells_per_block=(2, 2)),
    dict(orientations=12, pixels_per_cell=(8, 8),  cells_per_block=(2, 2)),
    dict(orientations=9,  pixels_per_cell=(4, 4),  cells_per_block=(2, 2)),
    dict(orientations=9,  pixels_per_cell=(8, 8),  cells_per_block=(3, 3)),
]

for cfg in configs:
    vec = extract_hog(patch, resize_to=(32, 32), **cfg)
    print(f"orientations={cfg['orientations']:2d}  "
          f"ppc={cfg['pixels_per_cell']}  "
          f"cpb={cfg['cells_per_block']}  "
          f"→ dim={vec.shape[0]}")

## 4. Feature variance across random patches

In [ ]:
patches = [rng.integers(0, 255, (32, 32), dtype=np.uint8) for _ in range(200)]
matrix = np.vstack([extract_hog(p, resize_to=(32, 32)) for p in patches])

variances = matrix.var(axis=0)
plt.figure(figsize=(10, 2))
plt.plot(variances)
plt.title('Per-dimension variance of HOG descriptors (random patches)')
plt.xlabel('Dimension')
plt.ylabel('Variance')
plt.tight_layout()
plt.show()